# Nifty50 Weekly Expiry — Hourly Candle Move Table

The same expiry-to-expiry question as `weekly_expiry_move_table.ipynb`, but measured on **hourly** candles instead of daily ones. Hourly resolution matters for option selling: a daily candle only tells you the day's high and low, while the hourly path tells you *when* a strike was breached and whether the index came back before expiry.

For every consecutive pair of weekly expiries this notebook measures, from the previous expiry's closing print:
- the max upside % and max downside % reached at any point during the week,
- the hour at which each extreme printed,
- the net close-to-close move realised at the next expiry,
- and how often a strike that was *touched* intraweek still expired worthless.

Source data:
- `data/raw/option_selling/nifty/nifty_weekly_expiry_dates.csv` — scheduled and actual weekly expiry dates
- `data/raw/option_selling/nifty/nifty50_2020-01-01_2026-09-15.json` — hourly OHLC candles (data starts 2022-01-03)

Set `SKIP_FIRST_CANDLES` in the parameters cell to drop the first N hourly candles after each expiry — the hourly equivalent of the `_skip_first_candle` variant of the daily notebook.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling" / "nifty"

EXPIRY_CSV = RAW_DIR / "nifty_weekly_expiry_dates.csv"
CANDLES_JSON = RAW_DIR / "nifty50_2020-01-01_2026-09-15.json"

# --- Parameters -------------------------------------------------------------
SKIP_FIRST_CANDLES = 0                                       # drop the first N hourly candles after each expiry
STRIKE_DISTANCES = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]  # % away from the prior expiry close
TOUCH_THRESHOLDS = [1.0, 2.0, 3.0]                           # thresholds tracked in the intraweek timing curve
REGULAR_SESSION_END_HOUR = 16                                # candles stamped later are Muhurat/special sessions
BIN_WIDTH, LOWER_TAIL, UPPER_TAIL = 1.0, -8.0, 6.0           # return-distribution buckets
# ---------------------------------------------------------------------------

# Diverging poles with a neutral midpoint: upside = warm, downside = cool.
UP_COLOR = (235, 104, 52)
DOWN_COLOR = (42, 120, 214)
NEUTRAL = (240, 239, 236)
UP_HEX, DOWN_HEX = "#eb6834", "#2a78d6"
SERIES_HEX = ["#2a78d6", "#eb6834", "#1baf7a"]   # fixed categorical order, never cycled
HEADER_FILL = "#2b2d42"

pd.options.display.float_format = "{:,.2f}".format

## Load Hourly Candles & Weekly Expiries

Expiry dates come from the exchange calendar, candles from the hourly dump. Two clean-ups happen here:

- **Muhurat / special sessions** (single candles stamped after 16:00) are dropped — they are not part of a normal expiry week.
- Each expiry date is **snapped back to the last session that actually traded on or before it**, so a holiday-shifted expiry still anchors on a real closing print. Duplicates after snapping are removed.

In [2]:
expiry_df = pd.read_csv(EXPIRY_CSV, parse_dates=["Actual_Expiry_Date"])
expiry_dates = expiry_df["Actual_Expiry_Date"].dt.normalize().sort_values().reset_index(drop=True)

with open(CANDLES_JSON) as f:
    raw = json.load(f)

price_df = pd.DataFrame(
    raw["data"]["candles"],
    columns=["timestamp", "open", "high", "low", "close", "volume", "oi"],
)
price_df["timestamp"] = (
    pd.to_datetime(price_df["timestamp"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
)
price_df = price_df.drop_duplicates("timestamp").set_index("timestamp").sort_index()
price_df = price_df[["open", "high", "low", "close"]].apply(pd.to_numeric)
price_df = price_df[price_df.index.hour < REGULAR_SESSION_END_HOUR]
price_df["session"] = price_df.index.normalize()

sessions = pd.DatetimeIndex(price_df["session"].unique())
session_close_ts = price_df.reset_index().groupby("session")["timestamp"].last()

in_range = expiry_dates[(expiry_dates >= sessions.min()) & (expiry_dates <= sessions.max())]
snapped = pd.Series(sessions[sessions.searchsorted(in_range, side="right") - 1])
expiry_sessions = snapped.drop_duplicates().reset_index(drop=True)
shifted_count = int((snapped.to_numpy() != in_range.to_numpy()).sum())

print(
    f"Loaded {len(price_df):,} hourly candles over {len(sessions):,} sessions, "
    f"{price_df.index.min():%Y-%m-%d %H:%M} to {price_df.index.max():%Y-%m-%d %H:%M}"
)
print(f"Candles per session: {price_df.groupby('session').size().median():.0f} (median)")
print(f"{len(expiry_sessions)} weekly expiries in range, {shifted_count} snapped back to the previous traded session")
display(price_df.head())

Loaded 8,127 hourly candles over 1,163 sessions, 2022-01-03 09:15 to 2026-09-15 15:15
Candles per session: 7 (median)
246 weekly expiries in range, 1 snapped back to the previous traded session


,open,high,low,close,session
timestamp,,,,,
2022-01-03 09:15:00,"17,387.15","17,501.65","17,387.15","17,498.10",2022-01-03
2022-01-03 10:15:00,"17,499.45","17,527.90","17,488.00","17,527.65",2022-01-03
2022-01-03 11:15:00,"17,528.35","17,534.00","17,512.00","17,525.35",2022-01-03
2022-01-03 12:15:00,"17,523.70","17,579.20","17,523.70","17,575.90",2022-01-03
2022-01-03 13:15:00,"17,575.85","17,613.85","17,567.95","17,611.00",2022-01-03


## Build Each Expiry-To-Expiry Window

Each window runs from the **previous expiry's last hourly close** (the anchor) through the last candle of the next expiry session:

- **base_close** — the anchor print; every % below is measured against it
- **max_up_pct** — highest hourly high in the window vs `base_close`
- **max_down_pct** — lowest hourly low in the window vs `base_close`
- **expiry_move_pct** — the close-to-close move actually settled at expiry
- **candles_to_max_up / down** — how many hourly candles into the week the extreme printed

`build_windows` also returns the hourly *path* of every week (running % vs the anchor), which the intraweek timing section consumes.

In [3]:
def build_windows(price: pd.DataFrame, expiries: pd.Series, skip_first: int = 0):
    """Measure every expiry-to-expiry window on hourly candles, anchored on the prior expiry close.

    Returns the per-week summary table and, for each week, the hourly (up %, down %) path
    relative to that week's anchor close.
    """
    rows, paths = [], []
    for prev_expiry, curr_expiry in zip(expiries.iloc[:-1], expiries.iloc[1:]):
        anchor_ts = session_close_ts.loc[prev_expiry]
        window = price[(price.index > anchor_ts) & (price["session"] <= curr_expiry)]
        if skip_first:
            window = window.iloc[skip_first:]
        if window.empty:
            continue

        base_close = price.at[anchor_ts, "close"]
        up_path = (window["high"] - base_close) / base_close * 100
        down_path = (window["low"] - base_close) / base_close * 100
        up_ts, down_ts = window["high"].idxmax(), window["low"].idxmin()
        expiry_close = window["close"].iloc[-1]

        rows.append(
            {
                "prev_expiry": prev_expiry,
                "curr_expiry": curr_expiry,
                "anchor_ts": anchor_ts,
                "base_close": base_close,
                "max_up_pct": up_path.max(),
                "max_up_ts": up_ts,
                "candles_to_max_up": window.index.get_loc(up_ts) + 1,
                "max_down_pct": down_path.min(),
                "max_down_ts": down_ts,
                "candles_to_max_down": window.index.get_loc(down_ts) + 1,
                "expiry_close": expiry_close,
                "expiry_move_pct": (expiry_close - base_close) / base_close * 100,
                "candles": len(window),
                "trading_days": window["session"].nunique(),
            }
        )
        paths.append(np.column_stack([up_path.to_numpy(), down_path.to_numpy()]))

    return pd.DataFrame(rows), paths


moves, paths = build_windows(price_df, expiry_sessions, SKIP_FIRST_CANDLES)
moves["year"] = moves["curr_expiry"].dt.year
moves["range_pct"] = moves["max_up_pct"] - moves["max_down_pct"]

print(f"{len(moves)} expiry-to-expiry weeks computed (skip_first_candles={SKIP_FIRST_CANDLES})")
display(moves.tail(10))

245 expiry-to-expiry weeks computed (skip_first_candles=0)


,prev_expiry,curr_expiry,anchor_ts,base_close,max_up_pct,max_up_ts,candles_to_max_up,max_down_pct,max_down_ts,candles_to_max_down,expiry_close,expiry_move_pct,candles,trading_days,year,range_pct
235,2026-07-07,2026-07-14,2026-07-07 15:15:00,"24,355.10",-0.23,2026-07-08 10:15:00,2,-2.26,2026-07-08 14:15:00,6,"24,035.15",-1.31,35,5,2026,2.03
236,2026-07-14,2026-07-21,2026-07-14 15:15:00,"24,035.15",1.38,2026-07-17 14:15:00,20,-0.10,2026-07-15 12:15:00,4,"24,193.05",0.66,35,5,2026,1.48
237,2026-07-21,2026-07-28,2026-07-21 15:15:00,"24,193.05",-0.11,2026-07-22 09:15:00,1,-2.43,2026-07-24 10:15:00,16,"23,987.80",-0.85,35,5,2026,2.31
238,2026-07-28,2026-08-04,2026-07-28 15:15:00,"23,987.80",3.28,2026-08-03 15:15:00,28,0.62,2026-07-29 09:15:00,1,"24,614.90",2.61,35,5,2026,2.66
239,2026-08-04,2026-08-11,2026-08-04 15:15:00,"24,614.90",0.25,2026-08-05 09:15:00,1,-0.75,2026-08-11 11:15:00,31,"24,471.70",-0.58,35,5,2026,1.01
240,2026-08-11,2026-08-18,2026-08-11 15:15:00,"24,471.70",0.01,2026-08-12 09:15:00,1,-1.29,2026-08-18 15:15:00,35,"24,154.90",-1.29,35,5,2026,1.30
241,2026-08-18,2026-08-25,2026-08-18 15:15:00,"24,154.90",0.74,2026-08-25 15:15:00,35,-0.54,2026-08-19 14:15:00,6,"24,334.55",0.74,35,5,2026,1.28
242,2026-08-25,2026-09-01,2026-08-25 15:15:00,"24,334.55",0.18,2026-08-26 09:15:00,1,-1.57,2026-09-01 14:15:00,34,"24,055.80",-1.15,35,5,2026,1.75
243,2026-09-01,2026-09-08,2026-09-01 15:15:00,"24,055.80",-0.13,2026-09-03 09:15:00,8,-1.80,2026-09-08 13:15:00,33,"23,635.10",-1.75,35,5,2026,1.67
244,2026-09-08,2026-09-15,2026-09-08 15:15:00,"23,635.10",-0.18,2026-09-15 09:15:00,22,-2.19,2026-09-15 15:15:00,28,"23,118.60",-2.19,28,4,2026,2.01


## Summary Stats

In [4]:
summary = pd.DataFrame(
    [
        ("Weeks analyzed", f"{len(moves)}"),
        ("Hourly candles per week (median)", f"{moves['candles'].median():.0f}"),
        ("Avg max up %", f"{moves['max_up_pct'].mean():.2f}%"),
        ("Avg max down %", f"{moves['max_down_pct'].mean():.2f}%"),
        ("Avg intraweek range (up − down)", f"{moves['range_pct'].mean():.2f}%"),
        (
            "Biggest single-week rally",
            f"{moves['max_up_pct'].max():.2f}% (expiry {moves.loc[moves['max_up_pct'].idxmax(), 'curr_expiry'].date()})",
        ),
        (
            "Biggest single-week crash",
            f"{moves['max_down_pct'].min():.2f}% (expiry {moves.loc[moves['max_down_pct'].idxmin(), 'curr_expiry'].date()})",
        ),
        ("Avg expiry-to-expiry close move", f"{moves['expiry_move_pct'].mean():.2f}%"),
        ("Weeks closed higher", f"{(moves['expiry_move_pct'] > 0).sum()} / {len(moves)}"),
        ("Weeks closed lower", f"{(moves['expiry_move_pct'] < 0).sum()} / {len(moves)}"),
        ("Median candle of the weekly high", f"{moves['candles_to_max_up'].median():.0f}"),
        ("Median candle of the weekly low", f"{moves['candles_to_max_down'].median():.0f}"),
    ],
    columns=["Metric", "Value"],
)
display(summary)

,Metric,Value
0,Weeks analyzed,245
1,Hourly candles per week (median),35
2,Avg max up %,1.31%
3,Avg max down %,-1.26%
4,Avg intraweek range (up − down),2.57%
5,Biggest single-week rally,6.48% (expiry 2025-04-17)
6,Biggest single-week crash,-6.95% (expiry 2022-06-16)
7,Avg expiry-to-expiry close move,0.13%
8,Weeks closed higher,133 / 245
9,Weeks closed lower,112 / 245


## Visual Table — Max Up % / Max Down % Per Expiry Week

Most recent week first. Shading encodes magnitude — deeper warm for a larger max upside, deeper cool for a larger max downside — and the signed value is printed in every cell, so direction never rests on colour alone. The *At* columns carry the hourly timestamp of each extreme.

In [5]:
def shade(values: pd.Series, color: tuple[int, int, int]) -> list[str]:
    """Blend from the neutral surface toward `color` in proportion to each value's magnitude."""
    magnitude = values.abs()
    peak = magnitude.max() or 1.0
    intensity = (magnitude / peak).clip(0, 1)
    return [
        "rgb({},{},{})".format(*[int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)])
        for t in intensity
    ]


table_df = moves.sort_values("curr_expiry", ascending=False).reset_index(drop=True)
white = ["white"] * len(table_df)
close_colors = [
    f"rgb{UP_COLOR}" if v > 0 else (f"rgb{DOWN_COLOR}" if v < 0 else f"rgb{NEUTRAL}")
    for v in table_df["expiry_move_pct"]
]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[85, 85, 80, 75, 120, 80, 120, 85],
            header=dict(
                values=[
                    "Prev Expiry", "Expiry", "Base Close", "Max Up %", "Max Up At",
                    "Max Down %", "Max Down At", "Expiry Move %",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=12),
                align="center",
                height=32,
            ),
            cells=dict(
                values=[
                    table_df["prev_expiry"].dt.strftime("%Y-%m-%d"),
                    table_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                    table_df["base_close"].map("{:,.2f}".format),
                    table_df["max_up_pct"].map("{:+.2f}%".format),
                    table_df["max_up_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                    table_df["max_down_pct"].map("{:+.2f}%".format),
                    table_df["max_down_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                    table_df["expiry_move_pct"].map("{:+.2f}%".format),
                ],
                fill_color=[
                    white, white, white,
                    shade(table_df["max_up_pct"], UP_COLOR),
                    white,
                    shade(table_df["max_down_pct"], DOWN_COLOR),
                    white,
                    close_colors,
                ],
                align="center",
                height=26,
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title="Nifty50 — Hourly Max Up / Max Down % From Prior Weekly Expiry",
    height=900,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## Touched vs Settled — The Option Seller's Ladder

For each distance from the prior expiry close, two very different probabilities:

- **Touched** — the index traded through that level at *some* hour during the week. This is what stops out a delta-hedged or stop-managed short.
- **Settled beyond** — the index was still past that level at expiry. This is what actually pays out on a held short.

The gap between the two columns is the premium a seller keeps for sitting through the noise; it is also the share of weeks where a stop-loss would have paid out on a strike that ultimately expired worthless. Hourly candles are what make the *touched* column trustworthy.

In [6]:
n_weeks = len(moves)
ladder = pd.DataFrame({"distance_pct": STRIKE_DISTANCES})
ladder["touch_up"] = [(moves["max_up_pct"] >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["touch_down"] = [(moves["max_down_pct"] <= -d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["settle_above"] = [(moves["expiry_move_pct"] >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["settle_below"] = [(moves["expiry_move_pct"] <= -d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["touched_either"] = [
    ((moves["max_up_pct"] >= d) | (moves["max_down_pct"] <= -d)).mean() * 100 for d in STRIKE_DISTANCES
]
ladder["settled_outside"] = [(moves["expiry_move_pct"].abs() >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["strangle_survived"] = 100 - ladder["settled_outside"]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[80, 80, 80, 85, 85, 95, 95, 105],
            header=dict(
                values=[
                    "Distance", "Touched<br>Up", "Touched<br>Down", "Settled<br>Above",
                    "Settled<br>Below", "Touched<br>Either Side", "Settled<br>Outside",
                    "Short Strangle<br>Expired Worthless",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=11),
                align="center",
                height=44,
            ),
            cells=dict(
                values=[
                    ladder["distance_pct"].map("±{:.1f}%".format),
                    ladder["touch_up"].map("{:.1f}%".format),
                    ladder["touch_down"].map("{:.1f}%".format),
                    ladder["settle_above"].map("{:.1f}%".format),
                    ladder["settle_below"].map("{:.1f}%".format),
                    ladder["touched_either"].map("{:.1f}%".format),
                    ladder["settled_outside"].map("{:.1f}%".format),
                    ladder["strangle_survived"].map("{:.1f}%".format),
                ],
                fill_color=[
                    ["#f4f3f0"] * len(ladder),
                    shade(ladder["touch_up"], UP_COLOR),
                    shade(ladder["touch_down"], DOWN_COLOR),
                    shade(ladder["settle_above"], UP_COLOR),
                    shade(ladder["settle_below"], DOWN_COLOR),
                    ["white"] * len(ladder),
                    ["white"] * len(ladder),
                    ["white"] * len(ladder),
                ],
                align="center",
                height=28,
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title=f"Touch vs Settle Probability By Distance From Prior Expiry Close ({n_weeks} weeks)",
    height=120 + 28 * len(ladder) + 60,
    margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

bars = go.Figure()
bars.add_trace(
    go.Bar(
        x=ladder["distance_pct"], y=ladder["touched_either"], name="Touched either side",
        marker_color=SERIES_HEX[0], marker_line=dict(color="white", width=2),
        text=ladder["touched_either"].map("{:.0f}%".format), textposition="outside",
    )
)
bars.add_trace(
    go.Bar(
        x=ladder["distance_pct"], y=ladder["settled_outside"], name="Settled outside at expiry",
        marker_color=SERIES_HEX[1], marker_line=dict(color="white", width=2),
        text=ladder["settled_outside"].map("{:.0f}%".format), textposition="outside",
    )
)
bars.update_layout(
    title="Path Risk vs Settlement Risk — How Often A Strike Is Touched But Not Breached At Expiry",
    barmode="group",
    bargap=0.25,
    template="plotly_white",
    height=440,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=20),
)
bars.update_xaxes(title_text="Strike distance from prior expiry close", ticksuffix="%", tickvals=STRIKE_DISTANCES)
bars.update_yaxes(title_text="% of weeks", ticksuffix="%", range=[0, 105])
bars.show()

## When In The Week Does The Damage Happen?

Each weekly path is walked hour by hour, tracking the share of weeks that had already traded ±1%, ±2% or ±3% away from the anchor by that point. A curve that is already flat by the middle of the week means the risk was front-loaded; one that keeps climbing into expiry means the last sessions are where shorts get tested.

Dotted verticals mark day boundaries (7 hourly candles per full session).

In [7]:
CANDLES_PER_DAY = 7
max_candles = max(len(p) for p in paths)

# Running |move| from the anchor: how far the week had travelled by candle k, either side.
excursion = np.full((len(paths), max_candles), np.nan)
for i, path in enumerate(paths):
    running = np.maximum.accumulate(np.maximum(path[:, 0], -path[:, 1]))
    excursion[i, : len(running)] = running

alive = (~np.isnan(excursion)).sum(axis=0)
x = np.arange(1, max_candles + 1)

fig = go.Figure()
for threshold, color in zip(TOUCH_THRESHOLDS, SERIES_HEX):
    touched = np.nansum(excursion >= threshold, axis=0) / np.maximum(alive, 1) * 100
    fig.add_trace(
        go.Scatter(
            x=x, y=touched, mode="lines", name=f"±{threshold:.0f}% touched",
            line=dict(color=color, width=2),
            hovertemplate="candle %{x} — %{y:.1f}% of weeks<extra></extra>",
        )
    )

for boundary in range(CANDLES_PER_DAY, max_candles, CANDLES_PER_DAY):
    fig.add_vline(x=boundary + 0.5, line_dash="dot", line_color="#c9c8c4", line_width=1)

fig.update_layout(
    title="Cumulative Share Of Weeks That Had Already Touched ±X% By Each Hourly Candle",
    template="plotly_white",
    height=460,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=20),
)
fig.update_xaxes(title_text="Hourly candles since the prior expiry close", gridcolor="#eceae6")
fig.update_yaxes(title_text="% of weeks touched", ticksuffix="%", gridcolor="#eceae6", rangemode="tozero")
fig.show()

timing = pd.DataFrame(
    {
        "Day of the expiry week": [f"Day {d}" for d in range(1, int(np.ceil(max_candles / CANDLES_PER_DAY)) + 1)],
        "Weekly high printed here": (
            ((moves["candles_to_max_up"] - 1) // CANDLES_PER_DAY + 1)
            .value_counts(normalize=True).sort_index().mul(100).round(1)
            .reindex(range(1, int(np.ceil(max_candles / CANDLES_PER_DAY)) + 1), fill_value=0.0).to_numpy()
        ),
        "Weekly low printed here": (
            ((moves["candles_to_max_down"] - 1) // CANDLES_PER_DAY + 1)
            .value_counts(normalize=True).sort_index().mul(100).round(1)
            .reindex(range(1, int(np.ceil(max_candles / CANDLES_PER_DAY)) + 1), fill_value=0.0).to_numpy()
        ),
    }
)
display(timing.style.format({"Weekly high printed here": "{:.1f}%", "Weekly low printed here": "{:.1f}%"}))

,Day of the expiry week,Weekly high printed here,Weekly low printed here
0,Day 1,26.5%,39.6%
1,Day 2,13.9%,14.7%
2,Day 3,13.5%,11.4%
3,Day 4,20.0%,16.7%
4,Day 5,25.3%,17.6%
5,Day 6,0.8%,0.0%


## Where The Volatility Sits — Hour Of Day × Weekday

Mean hourly candle range, `(high − low) / open`, across every regular session in the dataset. One hue, light to dark: darker cells are the hours that move most. This is the intraday texture behind the weekly numbers above — useful for choosing an entry or adjustment hour rather than a date.

In [8]:
intraday = price_df.copy()
intraday["weekday"] = intraday.index.day_name()
intraday["hour"] = intraday.index.strftime("%H:%M")
intraday["range_pct"] = (intraday["high"] - intraday["low"]) / intraday["open"] * 100

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
heat = (
    intraday[intraday["weekday"].isin(weekday_order)]
    .pivot_table(index="weekday", columns="hour", values="range_pct", aggfunc="mean")
    .reindex(weekday_order)
)

fig = go.Figure(
    data=go.Heatmap(
        z=heat.to_numpy(),
        x=heat.columns,
        y=heat.index,
        colorscale=[[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]],
        colorbar=dict(title="Mean range %", ticksuffix="%"),
        hovertemplate="%{y} %{x} — %{z:.2f}%<extra></extra>",
        xgap=2,
        ygap=2,
    )
)
fig.update_layout(
    title="Mean Hourly Candle Range By Weekday And Hour",
    template="plotly_white",
    height=380,
    margin=dict(t=60, b=50, l=90, r=20),
)
fig.update_xaxes(title_text="Candle start (IST)")
fig.show()

## Return Distribution — Expiry-To-Expiry Close Move %

Every weekly `expiry_move_pct` bucketed into 1% return ranges, ordered best to worst, with the probability of landing at or beyond each bound — the same shape as a broker's option-probability table. Colour runs warm for gains, cool for losses, neutral through the middle.

In [9]:
edges = np.arange(LOWER_TAIL, UPPER_TAIL + BIN_WIDTH, BIN_WIDTH)
bin_edges = [-np.inf, *edges, np.inf]

labels = [f"< {LOWER_TAIL:.0f}%"]
labels += [f"{lo:.0f}% to {hi:.0f}%" for lo, hi in zip(edges[:-1], edges[1:])]
labels.append(f"> {UPPER_TAIL:.0f}%")

bucket = pd.cut(moves["expiry_move_pct"], bins=bin_edges, labels=labels, right=False)
dist = bucket.value_counts().reindex(labels[::-1]).rename("periods").to_frame()   # best bucket first

total = int(dist["periods"].sum())
dist["pct_of_total"] = dist["periods"] / total * 100
dist["prob_at_least"] = dist["periods"].cumsum() / total * 100          # P(return >= bucket lower bound)
dist["prob_at_most"] = 100 - dist["prob_at_least"] + dist["pct_of_total"]  # P(return <= bucket upper bound)


def diverging(n: int) -> list[str]:
    """Warm (best) to neutral to cool (worst), matching the up/down poles used throughout."""
    colors = []
    for t in np.linspace(1.0, -1.0, n):
        pole = UP_COLOR if t >= 0 else DOWN_COLOR
        mix = abs(t)
        colors.append("rgb({},{},{})".format(*[int(NEUTRAL[i] + (pole[i] - NEUTRAL[i]) * mix) for i in range(3)]))
    return colors


row_colors = diverging(len(dist))

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[130, 110, 90, 140, 140],
            header=dict(
                values=[
                    "Return Range", "Number of Weeks", "% of Total",
                    "Probability (≥ lower bound)", "Probability (≤ upper bound)",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=12),
                align="center",
                height=34,
            ),
            cells=dict(
                values=[
                    dist.index,
                    dist["periods"],
                    dist["pct_of_total"].map("{:.1f}%".format),
                    dist["prob_at_least"].map("{:.1f}%".format),
                    dist["prob_at_most"].map("{:.1f}%".format),
                ],
                fill_color=[row_colors] * 5,
                font=dict(color="black", size=11),
                align="center",
                height=28,
            ),
        )
    ]
)
fig.update_layout(
    title=f"Weekly Expiry-To-Expiry Return Distribution ({total} weeks)",
    height=560,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## 🎯 Expiry Outliers & Global Events

Weeks whose expiry-to-expiry return fell outside a chosen probability interval. Drag the slider to change the coverage: the scatter, the counters and the table all re-render together. Outliers are the weeks where a short strangle sized off the average would have been run over.

In [10]:
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Markdown
from plotly.subplots import make_subplots

RETURN_COL = "expiry_move_pct"
WITHIN_COLOR = "#8a8880"

outlier_base = moves.sort_values("curr_expiry").reset_index(drop=True)


def classify_outliers(coverage_pct: int):
    tail = (100 - coverage_pct) / 2
    lower_q, upper_q = tail, 100 - tail
    lower_bound = np.percentile(outlier_base[RETURN_COL], lower_q)
    upper_bound = np.percentile(outlier_base[RETURN_COL], upper_q)

    df = outlier_base.copy()
    df["status"] = np.select(
        [df[RETURN_COL] < lower_bound, df[RETURN_COL] > upper_bound],
        ["Downside outlier", "Upside outlier"],
        default="Within boundary",
    )
    return df, lower_bound, upper_bound, lower_q, upper_q


def render_outliers(coverage_pct=80):
    df, lower_bound, upper_bound, lower_q, upper_q = classify_outliers(coverage_pct)
    total = len(df)
    n_down = int((df["status"] == "Downside outlier").sum())
    n_up = int((df["status"] == "Upside outlier").sum())
    n_out = n_down + n_up

    display(Markdown(
        f"**{total} weeks** &nbsp;|&nbsp; at **{coverage_pct}%** coverage the boundaries are the "
        f"**{lower_q:.1f}th** percentile ({lower_bound:+.2f}%) and the **{upper_q:.1f}th** "
        f"percentile ({upper_bound:+.2f}%)."
    ))

    color_map = {"Within boundary": WITHIN_COLOR, "Downside outlier": DOWN_HEX, "Upside outlier": UP_HEX}
    fig = go.Figure()
    for status, color in color_map.items():
        sub = df[df["status"] == status]
        fig.add_trace(
            go.Scatter(
                x=sub["curr_expiry"], y=sub[RETURN_COL], mode="markers", name=status,
                marker=dict(
                    color=color,
                    size=8 if status == "Within boundary" else 11,
                    line=dict(color="white", width=2),
                ),
                hovertemplate="%{x|%Y-%m-%d} — %{y:+.2f}%<extra></extra>",
            )
        )
    fig.add_hline(
        y=upper_bound, line_dash="dash", line_color=UP_HEX,
        annotation_text=f"Upper boundary ({upper_bound:+.2f}%)", annotation_position="top left",
    )
    fig.add_hline(
        y=lower_bound, line_dash="dash", line_color=DOWN_HEX,
        annotation_text=f"Lower boundary ({lower_bound:+.2f}%)", annotation_position="bottom left",
    )
    fig.update_layout(
        title="Weekly Expiry Returns — Inside And Outside The Chosen Boundary",
        template="plotly_white", height=460,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=80, b=50, l=60, r=20),
    )
    fig.update_xaxes(title_text="Expiry date", gridcolor="#eceae6")
    fig.update_yaxes(title_text="Expiry-to-expiry return", ticksuffix="%", gridcolor="#eceae6")
    fig.show()

    stat_fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
    stats = [
        ("Weeks", total, "#0b0b0b", ""),
        ("Outliers", n_out, "#0b0b0b", f" ({n_out / total * 100:.1f}%)" if total else ""),
        ("Downside outliers", n_down, DOWN_HEX, ""),
        ("Upside outliers", n_up, UP_HEX, ""),
    ]
    for i, (label, value, color, suffix) in enumerate(stats, start=1):
        stat_fig.add_trace(
            go.Indicator(
                mode="number", value=value,
                title={"text": label, "font": {"size": 13}},
                number={"font": {"size": 30, "color": color}, "suffix": suffix},
            ),
            row=1, col=i,
        )
    stat_fig.update_layout(height=140, template="plotly_white", margin=dict(t=10, b=10, l=10, r=10))
    stat_fig.show()

    out_df = df[df["status"] != "Within boundary"].copy()
    out_df = out_df.reindex(out_df[RETURN_COL].abs().sort_values(ascending=False).index)
    n_rows = len(out_df)
    status_colors = [DOWN_HEX if s == "Downside outlier" else UP_HEX for s in out_df["status"]]

    table_fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[90, 90, 95, 95, 130],
                header=dict(
                    values=["Prev Expiry", "Expiry", "Max Up %", "Max Down %", "Expiry Move %"],
                    fill_color=HEADER_FILL, font=dict(color="white", size=12), align="center", height=32,
                ),
                cells=dict(
                    values=[
                        out_df["prev_expiry"].dt.strftime("%Y-%m-%d"),
                        out_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                        out_df["max_up_pct"].map("{:+.2f}%".format),
                        out_df["max_down_pct"].map("{:+.2f}%".format),
                        out_df[RETURN_COL].map("{:+.2f}%".format),
                    ],
                    fill_color="white",
                    font=dict(color=[["black"] * n_rows] * 4 + [status_colors], size=11),
                    align="center", height=28,
                ),
            )
        ]
    )
    table_fig.update_layout(
        title="Outlier Weeks, Largest Absolute Move First",
        height=min(620, 90 + 28 * max(n_rows, 1)),
        margin=dict(t=50, b=10, l=10, r=10),
    )
    table_fig.show()


coverage_slider = widgets.IntSlider(
    value=80, min=50, max=99, step=1,
    description="Probability boundary coverage (%)",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
interact(render_outliers, coverage_pct=coverage_slider)

interactive(children=(IntSlider(value=80, continuous_update=False, description='Probability boundary coverage …

<function __main__.render_outliers(coverage_pct=80)>